# Ball detection cache (Colab)

Runs the soccer ball detector once over the five example clips and writes
`detections/<clip>.npz` in the format of the "Detection cache" section of
`research/ball_tracking/DESIGN.md`.

The pipeline matches `examples/soccer/main.py::run_ball_detection`
(`sv.InferenceSlicer`, 640×640 tiles, `OverlapFilter.NONE`, `imgsz=640`, then
`with_nms(threshold=0.1)`), except the confidence cutoff is lowered from the
ultralytics default 0.25 to **0.05**.

Use a GPU runtime (Runtime → Change runtime type → GPU). Expected time: ~30 min.
At the end, download `detections.zip` (or copy it to Drive) and commit the
`.npz` files to `research/ball_tracking/detections/`.

In [ ]:
!pip install -q gdown ultralytics supervision

## Download clips and the ball model

Same Google Drive files as `examples/soccer/setup.sh`. Nothing else is downloaded.

In [ ]:
import os

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

BALL_MODEL_ID = "1isw4wx-MK9h9LMr36VvIWlJD6ppUvw7V"
CLIP_IDS = {
    "08fd33_0": "1OG8K6wqUw9t7lp9ms1M48DxRhwTYciK-",
    "0bfacc_0": "12TqauVZ9tLAv8kWxTTBFWtgt2hNQ4_ZF",
    "121364_0": "1vVwjW1dE1drIdd4ZSILfbCGPD4weoNiu",
    "2e57b9_0": "19PGw55V8aA6GZu5-Aac5_9mCy3fNxmEf",
    "573e61_0": "1yYPKuXbHsCxqjA9G-S6aeR2Kcnos8RPU",
}

BALL_MODEL_PATH = os.path.join(DATA_DIR, "football-ball-detection.pt")
if not os.path.exists(BALL_MODEL_PATH):
    !gdown -O {BALL_MODEL_PATH} "https://drive.google.com/uc?id={BALL_MODEL_ID}"
for clip, file_id in CLIP_IDS.items():
    video_path = os.path.join(DATA_DIR, f"{clip}.mp4")
    if not os.path.exists(video_path):
        !gdown -O {video_path} "https://drive.google.com/uc?id={file_id}"

## Settings

In [ ]:
import hashlib
import json

import numpy as np
import supervision as sv
import torch
import ultralytics
from tqdm.auto import tqdm
from ultralytics import YOLO

CONFIDENCE_THRESHOLD = 0.05   # main.py uses the ultralytics default, 0.25
NMS_THRESHOLD = 0.1           # main.py: with_nms(threshold=0.1)
SLICE_WH = (640, 640)         # main.py: slice_wh=(640, 640)
IMGSZ = 640                   # main.py: imgsz=640
OUTPUT_DIR = "detections"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
print("supervision", sv.__version__, "| ultralytics", ultralytics.__version__)


def sha256(path: str) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


MODEL_SHA256 = sha256(BALL_MODEL_PATH)
print("model sha256:", MODEL_SHA256)

## Model and slicer

`main.py` passes `overlap_filter_strategy=sv.OverlapFilter.NONE`; newer
`supervision` releases renamed the argument to `overlap_filter`. Whichever the
installed version accepts is used, and every slicer attribute is recorded in
`meta` so the run is reproducible.

In [ ]:
import inspect

model = YOLO(BALL_MODEL_PATH).to(device=DEVICE)


def callback(image_slice: np.ndarray) -> sv.Detections:
    result = model(image_slice, imgsz=IMGSZ, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
    return sv.Detections.from_ultralytics(result)


slicer_params = inspect.signature(sv.InferenceSlicer.__init__).parameters
overlap_kwarg = "overlap_filter" if "overlap_filter" in slicer_params else "overlap_filter_strategy"
slicer = sv.InferenceSlicer(
    callback=callback,
    slice_wh=SLICE_WH,
    **{overlap_kwarg: sv.OverlapFilter.NONE},
)
SLICER_SETTINGS = {
    key: repr(value) for key, value in vars(slicer).items() if "callback" not in key
}
print(json.dumps(SLICER_SETTINGS, indent=2))

## Run detection and write caches

In [ ]:
def run_clip(clip: str) -> str:
    video_path = os.path.join(DATA_DIR, f"{clip}.mp4")
    video_info = sv.VideoInfo.from_video_path(video_path)
    frame_index, xyxy, confidence = [], [], []
    frame_count = 0
    for index, frame in enumerate(tqdm(
        sv.get_video_frames_generator(source_path=video_path),
        total=video_info.total_frames,
        desc=clip,
    )):
        detections = slicer(frame).with_nms(threshold=NMS_THRESHOLD)
        detections = detections[detections.confidence >= CONFIDENCE_THRESHOLD]
        frame_index.append(np.full(len(detections), index, dtype=np.int64))
        xyxy.append(detections.xyxy.astype(np.float32))
        confidence.append(detections.confidence.astype(np.float32))
        frame_count = index + 1

    meta = {
        "clip": clip,
        "fps": video_info.fps,
        "width": video_info.width,
        "height": video_info.height,
        "frame_count": frame_count,
        "header_frame_count": video_info.total_frames,
        "model_file": os.path.basename(BALL_MODEL_PATH),
        "model_sha256": MODEL_SHA256,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "nms_threshold": NMS_THRESHOLD,
        "imgsz": IMGSZ,
        "slicer": SLICER_SETTINGS,
        "supervision_version": sv.__version__,
        "ultralytics_version": ultralytics.__version__,
        "torch_version": torch.__version__,
        "device": DEVICE,
    }
    output_path = os.path.join(OUTPUT_DIR, f"{clip}.npz")
    tmp_path = output_path + ".tmp.npz"
    np.savez_compressed(
        tmp_path,
        frame_index=np.concatenate(frame_index),
        xyxy=np.concatenate(xyxy).reshape(-1, 4),
        confidence=np.concatenate(confidence),
        meta=np.array(json.dumps(meta, sort_keys=True)),
    )
    os.replace(tmp_path, output_path)
    return output_path


for clip in CLIP_IDS:
    print("wrote", run_clip(clip))

## Check the caches

In [ ]:
for clip in CLIP_IDS:
    with np.load(os.path.join(OUTPUT_DIR, f"{clip}.npz"), allow_pickle=False) as data:
        meta = json.loads(str(data["meta"]))
        n = len(data["frame_index"])
        per_frame = np.bincount(data["frame_index"], minlength=meta["frame_count"])
        print(
            f"{clip}: {meta['frame_count']} frames, {meta['width']}x{meta['height']}, "
            f"{n} candidates, {np.mean(per_frame > 0):.1%} frames with any, "
            f"{np.mean(data['confidence'] >= 0.25):.1%} of candidates at conf >= 0.25"
        )

## Download

Commit the files to `research/ball_tracking/detections/`.

In [ ]:
!cd {OUTPUT_DIR} && zip -q ../detections.zip *.npz
try:
    from google.colab import files
    files.download("detections.zip")
except ImportError:
    print("detections.zip written")